# Alert 생성기 실험 ⑤ — 룰만으로 Alert 만들기

생성기를 거치지 않는다. 룰이 고른 거래를 **대상별로 모은 것을 그대로 Alert 로** 만들고, 그 Alert 가 데이터셋의 **세탁 시도**(Patterns 파일의 BEGIN ~ END 한 덩어리)와 얼마나 같은지 본다.
지표는 02번과 같고, 모델 씨앗으로 생성기가 만든 Alert(01번)를 옆에 둔다.

| 방법 | 만드는 규칙 |
|---|---|
| G 생성기 | 01번 결과(`members.parquet`). 모델 씨앗 → 서비스 Alert 생성기 |
| R1 적중마다 | 룰 적중 묶음 하나가 Alert 하나. 같은 거래가 여러 Alert 에 들 수 있다 |
| R2 겹치면 합침 | 적중 묶음끼리 거래를 하나라도 공유하면 한 Alert 로 합친다. 한 거래는 한 Alert 에만 든다 |

**적중 묶음** = 한 룰이 같은 대상 때문에 고른 거래들 (03번 9-7절, `rule_lib.hit_groups`).
- FAN-OUT · FAN-IN · GATHER-SCATTER: 대상 계좌(뿌리는 / 모으는 / 가운데 계좌)가 같은 근거 거래
- SCATTER-GATHER: 도착 계좌가 같은 근거 거래
- 위 넷은 대상이 같아도, 이웃한 두 근거 거래의 간격이 그 룰의 기간을 넘으면 나눈다
- CYCLE · RANDOM: 근거 거래끼리 "이어지는 쌍"으로 연결된 덩어리

- 03번(9-7절까지)과 01번을 같은 DATASET 으로 먼저 돌려야 한다. 룰도 생성기도 여기서 다시 돌리지 않는다.
- 정답, 제외 패턴, "묶음은 거래 2건 이상" 규칙은 02번과 같다.
- 룰 적중 묶음은 읽은 기간 전체에서 만든 것이고, 여기서는 그중 **평가 구간 안의 거래**만 쓴다. 생성기도 평가 구간의 씨앗만 받았다.

## 0. 설정

In [1]:
DATASET = 'HI-Medium'            # 03번과 01번을 같은 DATASET 으로 먼저 돌려야 한다
RULE_RUN = 'rule_vs_model'               # 03번 결과 폴더 (hit_groups.parquet, hits.parquet)
GEN_RUN = 'before_after_grouping'        # 01번 결과 폴더 (members.parquet)
EXCLUDE = ['STACK', 'BIPARTITE']         # 평가에서 빼는 패턴 (02번과 같다)

In [2]:
from pathlib import Path
import json
import duckdb, numpy as np, pandas as pd
import gen_lib as GL
import rule_lib as RL

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'parquets').is_dir() or (p / 'datasets').is_dir()), None)
assert ROOT is not None, 'parquets/ 폴더를 못 찾음. 데이터가 있는 GPU 서버의 저장소 안에서 열어야 한다'
BASE = ROOT / 'parquets' / DATASET
RULE_DIR = Path.cwd() / 'results' / DATASET / RULE_RUN
GEN_DIR = Path.cwd() / 'results' / DATASET / GEN_RUN
OUT = Path.cwd() / 'results' / DATASET / 'rule_alerts'
OUT.mkdir(parents=True, exist_ok=True)
rule_run = json.load(open(RULE_DIR / 'run.json', encoding='utf-8'))
gen_run = json.load(open(GEN_DIR / 'run.json', encoding='utf-8'))
assert rule_run['test_window'] == gen_run['test_window'], '03번과 01번의 평가 구간이 다르다'
a, b = rule_run['test_window']
print('평가 구간', a, '~', b, '| 01번 점수', gen_run['scores'], '| 기준', gen_run['threshold'])
pd.DataFrame(rule_run['chosen']).T.rename(index=RL.RULES)      # 03번이 고른 룰 조건

평가 구간 2022-09-08 ~ 2022-09-16 | 01번 점수 ../../parquets/HI-Medium/service_scores/multi280_hgb36_v1_test_scores.parquet | 기준 0.194997844


,필터,반복,기간_분,조건값
FAN-OUT,첫거래+1천↑,허용,10080,6
FAN-IN,첫거래+환전,허용,10080,3
GATHER-SCATTER,첫거래+5천↑,—,10080,2
SCATTER-GATHER,1천↑,—,10080,4
CYCLE,없음,—,10080,3
RANDOM,(첫거래 또는 환전)+1천↑,—,4320,3


## 1. 룰 적중 묶음 읽기
`hit_groups.parquet` 의 한 줄은 (거래, 룰, 대상, 묶음)이다. 시각과 계좌는 거래 파일에서, 세탁 여부와 패턴은 03번의 `hits.parquet` 에서 붙인다.

In [3]:
groups = pd.read_parquet(RULE_DIR / 'hit_groups.parquet')
con = duckdb.connect()
con.register('group_ids', groups[['edge_id']].drop_duplicates())
tx = con.execute(f"""SELECT t.edge_id, t.ts, t.src, t.dst
                     FROM read_parquet('{BASE / 'transactions.parquet'}') t JOIN group_ids USING (edge_id)""").df()
groups = groups.merge(tx, on='edge_id', how='left')
assert groups.ts.notna().all(), '거래 파일에 없는 edge_id 가 있다'
start, end = pd.Timestamp(a), pd.Timestamp(b) + pd.Timedelta(days=1)
print(f'읽은 기간 전체: 거래 {groups.edge_id.nunique():,} 건, 적중 묶음 {groups[["룰", "묶음"]].drop_duplicates().shape[0]:,} 개')
groups = groups[(groups.ts >= start) & (groups.ts < end)]                    # 평가 구간 안의 거래만 쓴다

읽은 기간 전체: 거래 97,144 건, 적중 묶음 15,991 개


In [4]:
hits = pd.read_parquet(RULE_DIR / 'hits.parquet')
label = hits.set_index('edge_id')[['y', 'pattern_type']]
groups = groups.join(label, on='edge_id')
assert groups.y.notna().all(), 'hits.parquet 에 없는 거래가 있다. 03번을 9-7절까지 다시 돌려야 한다'
groups['y'] = groups.y.astype(int)
ORDER = [name for name in RL.RULES.values() if name in set(groups.룰)]
size = groups.groupby(['룰', '묶음']).size()
(size.groupby('룰').agg(적중_묶음='size', 거래='sum', 크기_중앙값='median', 크기_최대='max', 거래_1건짜리=lambda s: int((s == 1).sum()))
 .join(groups.groupby('룰').y.mean().round(4).rename('세탁_비율')).loc[ORDER])

,적중_묶음,거래,크기_중앙값,크기_최대,거래_1건짜리,세탁_비율
룰,,,,,,
FAN-OUT,354,2492,7.0,18,62,0.9623
FAN-IN,371,1917,4.0,15,20,0.9974
GATHER-SCATTER,212,1479,6.0,22,34,0.9919
SCATTER-GATHER,157,2103,10.0,59,6,0.9073
CYCLE,199,1020,3.0,33,37,0.8686
RANDOM,273,1355,4.0,17,21,0.8989


## 2. 정답: 세탁 시도
02번과 같은 방법이다. 시도마다 평가 구간 안의 거래 수를 세고, 구간 안 거래가 2건 이상인 시도를 평가한다.

In [5]:
PAT = BASE / 'patterns.parquet'
IN = f"ts >= TIMESTAMP '{start}' AND ts < TIMESTAMP '{end}'"
attempts = con.execute(f"""SELECT attempt_id, any_value(pattern_type) AS pattern_type, count(*) AS n_rows,
                                  count(*) FILTER (WHERE {IN}) AS n_in, count(edge_id) FILTER (WHERE {IN}) AS n_in_matched,
                                  min(ts) AS first_ts, max(ts) AS last_ts
                           FROM read_parquet('{PAT}') GROUP BY attempt_id""").df()
attempts['eligible'] = (~attempts.pattern_type.isin(EXCLUDE) & (attempts.n_in >= 2) & (attempts.n_in == attempts.n_in_matched))
# 거래 → 시도. 한 거래가 두 시도에 들어 있으면 번호가 작은 시도로 둔다
attempt_of = con.execute(f"""SELECT edge_id, min(attempt_id) AS attempt_id, arg_min(pattern_type, attempt_id) AS pattern_type,
                                    bool_and({IN}) AS in_window
                             FROM read_parquet('{PAT}') WHERE edge_id IS NOT NULL GROUP BY edge_id""").df()
attempt_of = attempt_of[~attempt_of.pattern_type.isin(EXCLUDE)]
eval_attempts = set(attempts.loc[attempts.eligible, 'attempt_id'])
truth = attempt_of[attempt_of.attempt_id.isin(eval_attempts) & attempt_of.in_window]      # 정답: 평가할 시도의 구간 안 거래
print(f'평가할 시도 {len(eval_attempts):,} 개 | 정답 거래 {len(truth):,} 건')

평가할 시도 1,291 개 | 정답 거래 8,897 건


In [6]:
# 묶기 전의 상한: 시도의 거래를 애초에 몇 건 골랐나. 고르지 못한 거래는 어떤 방법으로도 Alert 에 못 들어간다(생성기의 연결 거래 제외)
members = pd.read_parquet(GEN_DIR / 'members.parquet')
is_seed = members.seed_model | members.promoted          # 모델 씨앗 + 보강으로 들어온 씨앗 (02번과 같다)
picked = {'모델 씨앗': set(members.loc[is_seed, 'edge_id']), '룰이 고른 거래': set(groups.edge_id)}
rows = []
for name, ids in picked.items():
    t = truth.assign(hit=truth.edge_id.isin(ids)).groupby('attempt_id').agg(size=('hit', 'size'), hit=('hit', 'sum'))
    rows.append(dict(고른_것=name, 시도_수=len(t), 정답_거래=int(t['size'].sum()), 고른_거래=int(t.hit.sum()),
                     거래_재현율=round(float(t.hit.sum() / t['size'].sum()), 4), 전부_고른_시도=int((t.hit == t['size']).sum()),
                     두_건_이상_고른_시도=int((t.hit >= 2).sum()), 하나도_못_고른_시도=int((t.hit == 0).sum())))
ceiling = pd.DataFrame(rows).set_index('고른_것')
ceiling

,시도_수,정답_거래,고른_거래,거래_재현율,전부_고른_시도,두_건_이상_고른_시도,하나도_못_고른_시도
고른_것,,,,,,,
모델 씨앗,1291,8897,8767,0.9854,1187,1278,1
룰이 고른 거래,1291,8897,6675,0.7503,473,882,356


## 3. Alert 만들기
- **R1 적중마다**: (룰, 묶음) 하나가 Alert 하나다. GATHER-SCATTER 의 모으는 쪽을 FAN-IN 룰도 고르는 식으로, 같은 거래가 Alert 여러 개에 들 수 있다.
- **R2 겹치면 합침**: 거래를 하나라도 공유하는 적중 묶음을 한 Alert 로 합친다. 공유가 사슬처럼 이어지면 전부 한 Alert 가 된다.
- Alert 는 **거래 2건 이상**만 센다(02번과 같다). 평가 구간 안에 근거 거래가 1건뿐인 적중 묶음은 빠진다.
- 제외 패턴의 거래는 합친 뒤에 뺀다(02번이 생성기 결과를 다루는 방식과 같다).

In [7]:
groups['r1'] = pd.factorize(pd.MultiIndex.from_frame(groups[['룰', '묶음']]))[0]      # 적중 묶음 하나 = Alert 하나
groups['r2'] = RL.merge_groups(groups)                                              # 거래를 공유하는 적중 묶음끼리 합친 번호
kept = groups[~groups.pattern_type.isin(EXCLUDE)]

def as_groups(frame, column):
    g = frame[['edge_id', column, 'y']].drop_duplicates(['edge_id', column]).rename(columns={column: 'group'})
    return g[g.groupby('group')['edge_id'].transform('size') >= 2]

methods = {'G 생성기': as_groups(members[(members.role > 0) & ~members.pattern_type.isin(EXCLUDE)], 'alert'),
           'R1 적중마다': as_groups(kept, 'r1'), 'R2 겹치면 합침': as_groups(kept, 'r2')}
lonely = {'G 생성기': int((is_seed & ~members.pattern_type.isin(EXCLUDE)).sum()) - int(methods['G 생성기'].edge_id.isin(picked['모델 씨앗']).sum()),
          'R1 적중마다': kept.edge_id.nunique() - methods['R1 적중마다'].edge_id.nunique(),
          'R2 겹치면 합침': kept.edge_id.nunique() - methods['R2 겹치면 합침'].edge_id.nunique()}
print('고른 거래 중 Alert 에 못 든 것(혼자 남음):', lonely)
pd.DataFrame({name: dict(Alert=g.group.nunique(), 거래=g.edge_id.nunique()) for name, g in methods.items()})

고른 거래 중 Alert 에 못 든 것(혼자 남음): {'G 생성기': 1984, 'R1 적중마다': 106, 'R2 겹치면 합침': 106}


,G 생성기,R1 적중마다,R2 겹치면 합침
Alert,1259,1370,824
거래,9668,7198,7198


## 4. 정답과 비교

In [8]:
summaries, per_attempt = [], {}
for name, g in methods.items():
    summary, per = RL.compare_alerts(name, g, attempt_of, eval_attempts, truth)
    summaries.append(summary)
    per_attempt[name] = per
result = pd.DataFrame(summaries).set_index('방법').T.astype(object)   # 정수는 정수로 보이게
result

방법,G 생성기,R1 적중마다,R2 겹치면 합침
묶음_수,1259.0,1370.0,824.0
묶인_거래,9668.0,7198.0,7198.0
여러_묶음에_든_거래,0.0,2723.0,0.0
크기_중앙값,5.0,6.0,6.0
크기_최대,62.0,52.0,163.0
세탁_비율,0.9496,0.9296,0.9296
전부_정상_묶음,51.0,59.0,53.0
시도_2개이상_섞인_묶음,97.0,198.0,41.0
쌍_정밀도,0.6571,0.7699,0.4804
쌍_재현율,0.8832,0.6991,0.7566


In [9]:
# 확인: G 생성기 줄은 02번 결과와 같아야 한다 (02번을 돌려 둔 경우)
check = Path.cwd() / 'results' / DATASET / 'grouping_vs_truth' / 'summary.csv'
if check.exists():
    old = pd.read_csv(check, index_col=0)['G 생성기']
    diff = {k: (old[k], result.loc[k, 'G 생성기']) for k in old.index if k in result.index and abs(float(old[k]) - float(result.loc[k, 'G 생성기'])) > 1e-9}
    print('02번과 같다' if not diff else f'02번과 다른 항목: {diff}')

02번과 같다


**읽는 법** (02번과 같은 항목)
- `여러_묶음에_든_거래`: Alert 두 개 이상에 든 거래의 중복 수. R1 에서만 0 이 아니다.
- `세탁_비율`: Alert 에 든 거래 중 세탁 거래의 비율.
- `시도_2개이상_섞인_묶음`: 서로 다른 세탁 시도가 한 Alert 에 들어간 수.
- `쌍_정밀도`: 같은 Alert 에 든 두 거래가 실제로 같은 시도일 비율. 큰 Alert 에 이것저것 섞이면 떨어진다.
- `쌍_재현율`: 같은 시도의 두 거래가 같은 Alert 에 들어갔을 비율.
- `온전히_한_묶음`: 시도의 (구간 안) 거래 전부가 한 Alert 에 들어간 시도 수. 2절의 `전부_고른_시도` 가 상한이다.
- `평균_최대_포함률`: 시도마다 가장 많이 담은 Alert 하나가 그 시도 거래의 몇 %를 담았는지의 평균.
- `쪼개진_시도`: 시도의 거래가 Alert 2개 이상에 들어 있는 수. R1 에서는 같은 거래가 두 Alert 에 겹쳐 든 경우도 여기에 들어간다.
- `전혀_안_묶인_시도`: 시도의 거래가 어느 Alert 에도 안 든 수.

## 5. 패턴별

In [10]:
def table_by(keys, per_map=None):
    frames = []
    for name, per in (per_map or per_attempt).items():
        f = per.groupby(keys, observed=True).agg(시도=('size', 'size'), 온전=('coverage', lambda s: round(float((s == 1).mean()), 3)),
                                                 포함률=('coverage', lambda s: round(float(s.mean()), 3)),
                                                 쪼개짐=('n_groups', lambda s: round(float((s >= 2).mean()), 3)),
                                                 안_묶임=('covered', lambda s: round(float((s == 0).mean()), 3)))
        frames.append(f.add_prefix(name.split()[0] + '_'))
    return pd.concat(frames, axis=1)

by_pattern = table_by('pattern_type')
by_pattern

,G_시도,G_온전,G_포함률,G_쪼개짐,G_안_묶임,R1_시도,R1_온전,R1_포함률,R1_쪼개짐,R1_안_묶임,R2_시도,R2_온전,R2_포함률,R2_쪼개짐,R2_안_묶임
pattern_type,,,,,,,,,,,,,,,
CYCLE,236,0.758,0.849,0.127,0.064,236,0.513,0.582,0.335,0.347,236,0.521,0.593,0.013,0.347
FAN-IN,209,0.756,0.905,0.057,0.043,209,0.177,0.537,0.048,0.254,209,0.187,0.545,0.005,0.254
FAN-OUT,183,0.825,0.932,0.071,0.011,183,0.426,0.619,0.131,0.301,183,0.443,0.630,0.000,0.301
GATHER-SCATTER,267,0.476,0.789,0.199,0.064,267,0.124,0.550,0.449,0.247,267,0.240,0.606,0.011,0.247
RANDOM,168,0.833,0.910,0.107,0.024,168,0.411,0.597,0.107,0.292,168,0.452,0.611,0.030,0.292
SCATTER-GATHER,228,0.803,0.913,0.057,0.035,228,0.303,0.592,0.518,0.268,228,0.373,0.634,0.022,0.268


## 5-1. 시도 크기별
크기 = 평가 구간 안 거래 수.

In [11]:
SIZE_BINS, SIZE_LABELS = [1, 2, 5, 10, 20, 10_000], ['2건', '3~5건', '6~10건', '11~20건', '21건 이상']
by_size = table_by('크기', {name: per.assign(크기=pd.cut(per['size'], SIZE_BINS, labels=SIZE_LABELS)) for name, per in per_attempt.items()})
by_size

,G_시도,G_온전,G_포함률,G_쪼개짐,G_안_묶임,R1_시도,R1_온전,R1_포함률,R1_쪼개짐,R1_안_묶임,R2_시도,R2_온전,R2_포함률,R2_쪼개짐,R2_안_묶임
크기,,,,,,,,,,,,,,,
2건,291,0.660,0.744,0.069,0.172,291,0.182,0.234,0.076,0.715,291,0.196,0.241,0.007,0.715
3~5건,371,0.782,0.908,0.081,0.013,371,0.372,0.545,0.159,0.321,371,0.396,0.561,0.008,0.321
6~10건,376,0.750,0.921,0.122,0.000,376,0.396,0.737,0.330,0.088,376,0.447,0.760,0.024,0.088
11~20건,218,0.674,0.914,0.183,0.000,218,0.271,0.772,0.592,0.028,218,0.372,0.831,0.014,0.028
21건 이상,35,0.771,0.944,0.086,0.000,35,0.229,0.837,1.000,0.000,35,0.429,0.942,0.000,0.000


## 5-2. 평가 구간 뒤로 이어지는 시도 vs 구간 안에 끝난 시도
평가 구간이 끝난 뒤에도 거래가 이어지는 시도는 구간 안에서 **완성되기 전 모습**만 보인다. 룰은 완성된 모양을 찾기 때문에 여기서 더 불리하다.

In [12]:
after = attempts.set_index('attempt_id').last_ts >= end          # 시도 → 구간 뒤로 이어지는가
with_tail = {name: per.assign(끝=per.attempt_id.map(after).map(GL.TAIL_LABELS)) for name, per in per_attempt.items()}
by_tail = table_by('끝', with_tail)
by_tail

,G_시도,G_온전,G_포함률,G_쪼개짐,G_안_묶임,R1_시도,R1_온전,R1_포함률,R1_쪼개짐,R1_안_묶임,R2_시도,R2_온전,R2_포함률,R2_쪼개짐,R2_안_묶임
끝,,,,,,,,,,,,,,,
구간 뒤로 이어짐,451,0.716,0.893,0.120,0.027,451,0.197,0.491,0.224,0.355,451,0.255,0.527,0.009,0.355
구간 안에 끝남,840,0.732,0.868,0.101,0.051,840,0.379,0.623,0.319,0.245,840,0.420,0.644,0.015,0.245


In [13]:
# 패턴 × 끝
by_pattern_tail = table_by(['pattern_type', '끝'], with_tail)
by_pattern_tail

G_시도   G_온전  G_포함률  G_쪼개짐  G_안_묶임  R1_시도  R1_온전  \
pattern_type   끝                                                            
CYCLE          구간 뒤로 이어짐    69  0.812  0.885  0.087   0.043     69  0.377   
               구간 안에 끝남    167  0.737  0.835  0.144   0.072    167  0.569   
FAN-IN         구간 뒤로 이어짐    70  0.757  0.883  0.029   0.071     70  0.143   
               구간 안에 끝남    139  0.755  0.916  0.072   0.029    139  0.194   
FAN-OUT        구간 뒤로 이어짐    57  0.825  0.920  0.070   0.018     57  0.333   
               구간 안에 끝남    126  0.825  0.937  0.071   0.008    126  0.468   
GATHER-SCATTER 구간 뒤로 이어짐   145  0.510  0.872  0.234   0.000    145  0.069   
               구간 안에 끝남    122  0.434  0.691  0.156   0.139    122  0.189   
RANDOM         구간 뒤로 이어짐    45  0.867  0.892  0.044   0.067     45  0.400   
               구간 안에 끝남    123  0.821  0.917  0.130   0.008    123  0.415   
SCATTER-GATHER 구간 뒤로 이어짐    65  0.831  0.938  0.092   0.000     65  0.092   
               구간 안에 끝남    163  0.791  0.903  0.043   0.049    163  0.387   

                          R1_포함률  R1_쪼개짐  R1_안_묶임  R2_시도  R2_온전  R2_포함률  \
pattern_type   끝                                                          
CYCLE          구간 뒤로 이어짐   0.464   0.116    0.435     69  0.377   0.480   
               구간 안에 끝남    0.630   0.425    0.311    167  0.581   0.640   
FAN-IN         구간 뒤로 이어짐   0.420   0.043    0.429     70  0.157   0.430   
               구간 안에 끝남    0.596   0.050    0.165    139  0.201   0.603   
FAN-OUT        구간 뒤로 이어짐   0.468   0.070    0.474     57  0.351   0.478   
               구간 안에 끝남    0.687   0.159    0.222    126  0.484   0.698   
GATHER-SCATTER 구간 뒤로 이어짐   0.561   0.393    0.193    145  0.166   0.615   
               구간 안에 끝남    0.537   0.516    0.311    122  0.328   0.596   
RANDOM         구간 뒤로 이어짐   0.484   0.111    0.467     45  0.444   0.502   
               구간 안에 끝남    0.639   0.106    0.228    123  0.455   0.651   
SCATTER-GATHER 구간 뒤로 이어짐   0.466   0.369    0.369     65  0.215   0.542   
               구간 안에 끝남    0.642   0.577    0.227    163  0.436   0.670   

                          R2_쪼개짐  R2_안_묶임  
pattern_type   끝                           
CYCLE          구간 뒤로 이어짐   0.014    0.435  
               구간 안에 끝남    0.012    0.311  
FAN-IN         구간 뒤로 이어짐   0.000    0.429  
               구간 안에 끝남    0.007    0.165  
FAN-OUT        구간 뒤로 이어짐   0.000    0.474  
               구간 안에 끝남    0.000    0.222  
GATHER-SCATTER 구간 뒤로 이어짐   0.014    0.193  
               구간 안에 끝남    0.008    0.311  
RANDOM         구간 뒤로 이어짐   0.022    0.467  
               구간 안에 끝남    0.033    0.228  
SCATTER-GATHER 구간 뒤로 이어짐   0.000    0.369  
               구간 안에 끝남    0.031    0.227

## 6. 룰 Alert 들여다보기
### 6-1. 합친 Alert(R2)는 어떤 룰들로 이루어졌나

In [14]:
r2 = kept[kept.r2.isin(methods['R2 겹치면 합침'].group)]
alerts = r2.groupby('r2').agg(거래=('edge_id', 'nunique'), 적중_묶음=('r1', 'nunique'), 룰_수=('룰', 'nunique'),
                              룰=('룰', lambda s: ' + '.join(name for name in ORDER if name in set(s))))
alerts = alerts.join(r2.drop_duplicates(['r2', 'edge_id']).groupby('r2').y.sum().rename('세탁'))
alerts = alerts.join(r2.merge(attempt_of[['edge_id', 'attempt_id']], on='edge_id').groupby('r2').attempt_id.nunique().rename('시도_수')).fillna({'시도_수': 0})
alerts['시도_수'] = alerts.시도_수.astype(int)
alerts.groupby('룰_수').agg(Alert=('거래', 'size'), 거래_합=('거래', 'sum'), 크기_중앙값=('거래', 'median'), 크기_최대=('거래', 'max'),
                            세탁_비율=('세탁', lambda s: round(float(s.sum() / alerts.loc[s.index, '거래'].sum()), 4)),
                            시도_2개이상=('시도_수', lambda s: int((s >= 2).sum())))

,Alert,거래_합,크기_중앙값,크기_최대,세탁_비율,시도_2개이상
룰_수,,,,,,
1,541,3026,5.0,22,0.9230,17
2,168,1530,8.0,45,0.9170,6
3,103,1916,18.0,36,0.9374,8
4,3,72,20.0,34,0.9167,1
5,5,218,48.0,85,0.9633,5
6,4,436,99.0,163,0.9702,4


In [15]:
# 어떤 룰 조합이 많은가 (위 10개)
combo = alerts.groupby('룰').agg(Alert=('거래', 'size'), 거래_합=('거래', 'sum'), 크기_중앙값=('거래', 'median'),
                                 세탁_비율=('세탁', lambda s: round(float(s.sum() / alerts.loc[s.index, '거래'].sum()), 4)),
                                 시도_2개이상=('시도_수', lambda s: int((s >= 2).sum()))).sort_values('Alert', ascending=False)
combo.head(10)

,Alert,거래_합,크기_중앙값,세탁_비율,시도_2개이상
룰,,,,,
FAN-IN,185,998,4.0,0.9970,0
RANDOM,153,817,4.0,0.8862,0
FAN-OUT,107,825,7.0,0.9648,0
CYCLE,74,295,3.0,0.6712,16
CYCLE + RANDOM,62,403,6.0,0.9355,1
FAN-OUT + FAN-IN + SCATTER-GATHER,56,1206,20.5,0.9212,2
FAN-OUT + FAN-IN + GATHER-SCATTER,38,575,15.0,0.9722,2
FAN-IN + GATHER-SCATTER,31,374,12.0,1.0000,0
FAN-OUT + GATHER-SCATTER,26,244,9.0,0.9836,0


### 6-2. 큰 Alert
방법마다 가장 큰 Alert 5개. 합치는 방식(R2)은 계좌를 타고 이어지면 커질 수 있다.

In [16]:
rows = []
for name, g in methods.items():
    m = g.merge(attempt_of[['edge_id', 'attempt_id']], on='edge_id', how='left')
    stat = m.groupby('group').agg(거래=('edge_id', 'size'), 세탁=('y', 'sum'), 시도_수=('attempt_id', 'nunique'))
    for group, r in stat.sort_values('거래', ascending=False).head(5).iterrows():
        rows.append(dict(방법=name, 거래=int(r.거래), 세탁=int(r.세탁), 시도_수=int(r.시도_수),
                         룰=alerts.loc[group, '룰'] if name.startswith('R2') else ''))
largest = pd.DataFrame(rows)
largest

,방법,거래,세탁,시도_수,룰
0,G 생성기,62,62,10,
1,G 생성기,62,62,15,
2,G 생성기,61,60,13,
3,G 생성기,57,56,19,
4,G 생성기,54,54,13,
5,R1 적중마다,52,50,12,
6,R1 적중마다,44,0,0,
7,R1 적중마다,38,38,7,
8,R1 적중마다,33,30,1,
9,R1 적중마다,32,30,1,


## 7. 생성기 Alert 와 겹침
룰 Alert(R2)가 모델 씨앗 · 생성기 Alert 와 얼마나 겹치는지 본다. 모델 씨앗이 하나도 없는 룰 Alert 가 **룰만으로 새로 생기는 Alert** 다.

In [17]:
gen = methods['G 생성기']
alert_tx = methods['R2 겹치면 합침'].copy()
alert_tx['모델_씨앗'] = alert_tx.edge_id.isin(picked['모델 씨앗'])
alert_tx['생성기_Alert'] = alert_tx.edge_id.map(gen.drop_duplicates('edge_id').set_index('edge_id').group)
per_alert = alert_tx.groupby('group').agg(거래=('edge_id', 'size'), 세탁=('y', 'sum'), 모델_씨앗=('모델_씨앗', 'sum'),
                                          걸친_생성기_Alert=('생성기_Alert', 'nunique'))
per_alert['모델과'] = np.select([per_alert.모델_씨앗 == 0, per_alert.모델_씨앗 == per_alert.거래], ['씨앗 없음 (룰만)', '전부 씨앗'], '일부 씨앗')
overlap = per_alert.groupby('모델과').agg(Alert=('거래', 'size'), 거래=('거래', 'sum'), 세탁=('세탁', 'sum'),
                                          세탁_0건_Alert=('세탁', lambda s: int((s == 0).sum())),
                                          생성기_Alert_2개이상에_걸침=('걸친_생성기_Alert', lambda s: int((s >= 2).sum())))
overlap['세탁_비율'] = (overlap.세탁 / overlap.거래).round(4)
overlap

,Alert,거래,세탁,세탁_0건_Alert,생성기_Alert_2개이상에_걸침,세탁_비율
모델과,,,,,,
씨앗 없음 (룰만),55,247,6,51,0,0.0243
일부 씨앗,78,1252,1118,0,8,0.8930
전부 씨앗,691,5699,5567,2,57,0.9768


In [18]:
# 거래 단위: 두 방법의 Alert 에 든 거래가 얼마나 같은가
rule_ids, gen_ids = set(alert_tx.edge_id), set(gen.edge_id)
y_of = pd.concat([alert_tx[['edge_id', 'y']], gen[['edge_id', 'y']]]).drop_duplicates('edge_id').set_index('edge_id').y
tx_overlap = pd.DataFrame([dict(거래=name, 건수=len(ids), 세탁=int(y_of.loc[list(ids)].sum()),
                                세탁_비율=round(float(y_of.loc[list(ids)].mean()), 4) if ids else None)
                           for name, ids in [('둘 다의 Alert 에 든 거래', rule_ids & gen_ids), ('룰 Alert 에만', rule_ids - gen_ids),
                                             ('생성기 Alert 에만', gen_ids - rule_ids)]]).set_index('거래')
tx_overlap

,건수,세탁,세탁_비율
거래,,,
둘 다의 Alert 에 든 거래,6701,6544,0.9766
룰 Alert 에만,497,147,0.2958
생성기 Alert 에만,2967,2637,0.8888


## 8. 저장

In [19]:
result.to_csv(OUT / 'summary.csv')
ceiling.to_csv(OUT / 'ceiling.csv')
by_pattern.to_csv(OUT / 'by_pattern.csv'); by_size.to_csv(OUT / 'by_size.csv')
by_tail.to_csv(OUT / 'by_tail.csv'); by_pattern_tail.to_csv(OUT / 'by_pattern_tail.csv')
alerts.to_csv(OUT / 'alerts.csv'); combo.to_csv(OUT / 'rule_combos.csv'); largest.to_csv(OUT / 'largest.csv', index=False)
overlap.to_csv(OUT / 'overlap_alerts.csv'); tx_overlap.to_csv(OUT / 'overlap_tx.csv')
pd.concat([per.assign(method=name) for name, per in per_attempt.items()]).to_csv(OUT / 'per_attempt.csv', index=False)
groups[['edge_id', '룰', '대상', '묶음', 'r1', 'r2', 'y', 'pattern_type']].to_parquet(OUT / 'members.parquet', index=False)      # 룰 Alert 구성 거래
json.dump(dict(dataset=DATASET, rule_run=RULE_RUN, gen_run=GEN_RUN, test_window=[a, b], exclude=EXCLUDE,
               chosen=rule_run['chosen'], lonely=lonely),
          open(OUT / 'run.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print(sorted(p.name for p in OUT.iterdir()))

['alerts.csv', 'by_pattern.csv', 'by_pattern_tail.csv', 'by_size.csv', 'by_tail.csv', 'ceiling.csv', 'largest.csv', 'members.parquet', 'overlap_alerts.csv', 'overlap_tx.csv', 'per_attempt.csv', 'rule_combos.csv', 'run.json', 'summary.csv']
